# AIAT 125 — Deploying AI Models
## Final Comprehensive Exercise — INSTRUCTOR SOLUTION KEY

**Course**: AIAT 125 | **Institution**: Tuwaiq Academy for Training | **Total Points**: 100

> **This is the reference solution. Do not distribute to students.**

---

| Part | CLO | Topic | Points |
|------|-----|-------|--------|
| Part 1 | CLO1 | Deployment Lifecycle + Model Training | 10 |
| Part 2 | CLO2 | Model Packaging (Pickle/Joblib/ONNX) + Serving Frameworks | 15 |
| Part 3 | CLO3 | REST APIs — FastAPI + Flask | 25 |
| Part 4 | CLO4 | Cloud Deployment (AWS/GCP/Azure) + Security | 10 |
| Part 5 | CLO5 | Docker + Kubernetes (Deployment/Service/HPA) + CI/CD | 15 |
| Part 6 | CLO6 | Monitoring + Alerting + Drift + MLflow/WandB + Versioning + Retraining + A/B + Canary | 25 |
| **Total** | | | **100** |

In [ ]:
# ── SETUP — Run this cell first ───────────────────────────────────────────────
import subprocess, sys
pkgs = [
    "scikit-learn", "numpy", "pandas", "scipy", "joblib",
    "fastapi", "flask", "httpx", "pydantic", "mlflow",
    "skl2onnx", "onnxruntime"
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs, check=False)

import os, json, time, pickle, warnings
import numpy as np
import pandas as pd
import joblib
from datetime import datetime
from scipy import stats
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
warnings.filterwarnings('ignore')

BASE_DIR = "/tmp/aiat125_final/"
os.makedirs(BASE_DIR, exist_ok=True)

np.random.seed(42)
X_raw, y_raw = make_classification(
    n_samples=1000, n_features=6, n_informative=4, n_redundant=2, random_state=42
)
FEATURE_NAMES = ["age_norm","bp_norm","glucose_norm","bmi_norm","feature_5","feature_6"]
CLASS_NAMES   = ["low_risk", "high_risk"]

X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42
)

print(f"Setup complete | Train={len(X_train)} Test={len(X_test)} Features={len(FEATURE_NAMES)}")
print(f"Save dir: {BASE_DIR}")

---
# Part 1 — Deployment Lifecycle (CLO1) — 10 Points

## Task 1A — Lifecycle Stages Knowledge (5 points)

In [ ]:
LIFECYCLE_STAGES = {
    "development": 1,
    "testing":     2,
    "packaging":   3,
    "deployment":  4,
    "monitoring":  5,
    "retraining":  6,
}

STAGE_DESCRIPTIONS = {
    "development": "Train and evaluate the model using labeled data to meet accuracy requirements.",
    "testing":     "Validate model performance against acceptance criteria before production release.",
    "packaging":   "Serialize the model and bundle all dependencies for portable deployment.",
    "deployment":  "Serve the packaged model via an API endpoint in the production environment.",
    "monitoring":  "Track live performance metrics, detect data drift, and log predictions.",
    "retraining":  "Trigger a new training run when performance degrades or distribution shift is detected.",
}

FEEDBACK_LOOP_DEFINITION = "The feedback loop uses monitoring data from production to detect performance degradation and trigger retraining, closing the cycle from deployment back to development."

for stage, order in sorted(LIFECYCLE_STAGES.items(), key=lambda x: x[1]):
    print(f"  Stage {order}: {stage:15s} — {STAGE_DESCRIPTIONS.get(stage)}")

In [ ]:
assert sorted(LIFECYCLE_STAGES.values()) == [1,2,3,4,5,6], "Each stage needs a unique number from 1 to 6"
assert LIFECYCLE_STAGES["development"] < LIFECYCLE_STAGES["deployment"]
assert LIFECYCLE_STAGES["deployment"]  < LIFECYCLE_STAGES["monitoring"]
assert LIFECYCLE_STAGES["monitoring"]  < LIFECYCLE_STAGES["retraining"]
assert all(v and len(v)>5 for v in STAGE_DESCRIPTIONS.values()), "Describe each stage"
assert FEEDBACK_LOOP_DEFINITION and len(FEEDBACK_LOOP_DEFINITION)>10
print("Task 1A PASSED (5/5 pts)")

---
## Task 1B — Train, Validate and Prepare Model (5 points)

In [ ]:
production_model = RandomForestClassifier(n_estimators=100, random_state=42)
production_model.fit(X_train, y_train)

y_pred        = production_model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)

DEPLOY_THRESHOLD = 0.80
deployment_ready = bool(test_accuracy >= DEPLOY_THRESHOLD)

deployment_report = {
    "model_type":    "RandomForestClassifier",
    "test_accuracy": test_accuracy,
    "threshold":     DEPLOY_THRESHOLD,
    "ready":         deployment_ready,
    "timestamp":     datetime.utcnow().isoformat(),
}

print(f"Accuracy={test_accuracy:.4f} | Ready={deployment_ready}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier as RFC
assert isinstance(production_model, RFC) and production_model.n_estimators == 100
assert test_accuracy >= DEPLOY_THRESHOLD, f"accuracy={test_accuracy:.4f} below threshold"
assert deployment_ready, "deployment_ready must be True"
for k in ["model_type","test_accuracy","threshold","ready","timestamp"]:
    assert k in deployment_report, f"deployment_report missing '{k}'"
print(f"Task 1B PASSED — accuracy={test_accuracy:.4f} (5/5 pts)")

---
# Part 2 — Model Packaging & Serialization (CLO2) — 15 Points

## Task 2A — Pickle + Joblib Packaging (4 points)

In [ ]:
PICKLE_PATH = os.path.join(BASE_DIR, "model.pkl")
JOBLIB_PATH = os.path.join(BASE_DIR, "model_bundle.joblib")
META_PATH   = os.path.join(BASE_DIR, "model_metadata.json")

# 2A-i: Save with pickle
with open(PICKLE_PATH, "wb") as f:
    pickle.dump(production_model, f)

# 2A-ii: Bundle and save with joblib
model_bundle = {
    "model":         production_model,
    "feature_names": FEATURE_NAMES,
    "class_names":   CLASS_NAMES,
    "metadata": {
        "version":    "v1.0.0",
        "accuracy":   test_accuracy,
        "framework":  "sklearn",
        "trained_at": datetime.utcnow().isoformat(),
    },
}
joblib.dump(model_bundle, JOBLIB_PATH)

# 2A-iii: Save metadata only (no model)
with open(META_PATH, "w") as f:
    json.dump(model_bundle["metadata"], f)

# 2A-iv: Reload
loaded_bundle   = joblib.load(JOBLIB_PATH)
loaded_model    = loaded_bundle["model"]
loaded_metadata = loaded_bundle["metadata"]

print(f"Pickle  : {os.path.getsize(PICKLE_PATH):,} bytes")
print(f"Joblib  : {os.path.getsize(JOBLIB_PATH):,} bytes")
print(f"Metadata: {loaded_metadata}")

In [ ]:
assert os.path.exists(PICKLE_PATH)
with open(PICKLE_PATH,'rb') as f: assert isinstance(pickle.load(f), RFC)
assert os.path.exists(JOBLIB_PATH)
assert isinstance(model_bundle, dict)
for k in ["model","feature_names","class_names","metadata"]: assert k in model_bundle
for k in ["version","accuracy","framework","trained_at"]: assert k in model_bundle["metadata"]
assert os.path.exists(META_PATH)
assert "model" not in json.loads(open(META_PATH).read()), "metadata JSON must not contain the model"
assert loaded_model is not None and loaded_metadata is not None
_acc = accuracy_score(y_test, loaded_model.predict(X_test))
assert abs(_acc - test_accuracy) < 0.001, "The loaded model must produce the same results"
print("Task 2A PASSED (4/4 pts)")

---
## Task 2B — ONNX Format: Convert, Save, and Run (5 points)

In [ ]:
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
import onnxruntime as rt

ONNX_PATH = os.path.join(BASE_DIR, "model.onnx")

# 2B-i: Convert and save
initial_type = [("float_input", FloatTensorType([None, 6]))]
onnx_model   = convert_sklearn(production_model, initial_types=initial_type)
with open(ONNX_PATH, "wb") as f:
    f.write(onnx_model.SerializeToString())

# 2B-ii: Load and run inference
sess       = rt.InferenceSession(ONNX_PATH)
input_name = sess.get_inputs()[0].name
onnx_preds = sess.run(None, {input_name: X_test[:5].astype(np.float32)})[0]

# 2B-iii: Conceptual answers
Q_ONNX_WHEN      = "Use ONNX when deploying across different frameworks or runtimes, such as running a scikit-learn model in a C++ service or on a mobile device."
Q_ONNX_ADVANTAGE = "onnxruntime is optimized for inference speed and runs on CPU/GPU without the full scikit-learn installation, making deployments lighter and faster."

print(f"ONNX file size : {os.path.getsize(ONNX_PATH):,} bytes")
print(f"ONNX preds (5) : {onnx_preds[:5]}")
print(f"sklearn preds  : {production_model.predict(X_test[:5])}")

In [ ]:
assert os.path.exists(ONNX_PATH), "ONNX file not found"
assert os.path.getsize(ONNX_PATH) > 1000, "ONNX file is too small — check conversion"
assert sess is not None, "sess must be an InferenceSession"
assert onnx_preds is not None, "onnx_preds must not be None"
assert len(onnx_preds) == 5, "Number of predictions must be 5"
_sk_preds = production_model.predict(X_test[:5])
assert np.array_equal(np.array(onnx_preds).flatten()[:5], _sk_preds), \
    "ONNX predictions must match sklearn predictions"
assert Q_ONNX_WHEN and len(Q_ONNX_WHEN) > 10
assert Q_ONNX_ADVANTAGE and len(Q_ONNX_ADVANTAGE) > 10
print("Task 2B PASSED — ONNX converted, saved, and verified (5/5 pts)")

---
## Task 2C — Serving Frameworks: TF Serving & TorchServe (3 points)

In [ ]:
TF_SERVING_CONFIG = """
model_config_list {
  config {
    name: "patient_risk_classifier"
    base_path: "/models/patient_risk_classifier"
    model_platform: "tensorflow"
  }
}
"""

TORCHSERVE_MANIFEST = {
    "model_name":      "patient_risk_classifier",
    "handler":         "custom_handler.py",
    "serialized_file": "model.pt",
    "model_file":      "model.py",
    "version":         "1.0",
}

Q_TF_VS_FLASK  = "TF Serving provides built-in versioning, batching, and gRPC support without any serving code, while Flask requires manual implementation of all these features."
Q_MULTI_MODEL  = "Multi-model serving lets you deploy and manage multiple models behind one server, reducing infrastructure costs and simplifying routing between model versions."

with open(os.path.join(BASE_DIR, "tf_serving.config"), "w") as f:
    f.write(TF_SERVING_CONFIG)
print(TF_SERVING_CONFIG)
print(json.dumps(TORCHSERVE_MANIFEST, indent=2))

In [ ]:
assert TF_SERVING_CONFIG and isinstance(TF_SERVING_CONFIG, str)
tf = TF_SERVING_CONFIG.lower()
assert "model_config_list" in tf or "model_config" in tf
assert "name" in tf
assert "base_path" in tf
assert "model_platform" in tf or "tensorflow" in tf
assert isinstance(TORCHSERVE_MANIFEST, dict)
for k in ["model_name","handler","serialized_file","model_file","version"]:
    assert k in TORCHSERVE_MANIFEST, f"TORCHSERVE_MANIFEST missing '{k}'"
assert Q_TF_VS_FLASK and len(Q_TF_VS_FLASK) > 10
assert Q_MULTI_MODEL and len(Q_MULTI_MODEL) > 10
print("Task 2C PASSED — TF Serving config + TorchServe manifest (3/3 pts)")

---
## Task 2D — Batch vs Real-Time Predict Functions (3 points)

In [ ]:
def predict_single(model, features):
    t0 = time.perf_counter()
    if len(features) != 6:
        raise ValueError("features must have exactly 6 elements")
    try:
        arr = [float(f) for f in features]
    except (TypeError, ValueError):
        raise ValueError("all features must be numeric")
    X     = np.array([arr])
    proba = model.predict_proba(X)[0]
    class_id   = int(np.argmax(proba))
    latency_ms = (time.perf_counter() - t0) * 1000
    return {
        "prediction": CLASS_NAMES[class_id],
        "confidence": round(float(proba.max()), 4),
        "latency_ms": round(latency_ms, 4),
    }


def predict_batch(model, data):
    t0   = time.perf_counter()
    data = np.array(data)
    if len(data) == 0:
        raise ValueError("data must not be empty")
    if data.ndim != 2 or data.shape[1] != 6:
        raise ValueError("each row must have exactly 6 columns")
    probas     = model.predict_proba(data)
    class_ids  = np.argmax(probas, axis=1)
    total_time = (time.perf_counter() - t0) * 1000
    return {
        "predictions":   [CLASS_NAMES[c] for c in class_ids],
        "confidences":   [round(float(p.max()), 4) for p in probas],
        "count":         len(data),
        "total_time_ms": round(total_time, 4),
    }


print("Single:", predict_single(production_model, X_test[0].tolist()))
print("Batch: ", predict_batch(production_model, X_test[:5]))

In [ ]:
try: predict_single(production_model, [1,2,3]); assert False
except ValueError: pass
try: predict_single(production_model, [1,2,"x",0,1,0]); assert False
except ValueError: pass
_r = predict_single(production_model, X_test[0].tolist())
assert set(_r.keys()) == {"prediction","confidence","latency_ms"}
assert _r["prediction"] in CLASS_NAMES
assert 0 <= _r["confidence"] <= 1
try: predict_batch(production_model, []); assert False
except ValueError: pass
try: predict_batch(production_model, [[1,2]]); assert False
except ValueError: pass
_b = predict_batch(production_model, X_test[:10])
assert _b["count"] == 10
assert len(_b["predictions"]) == 10
assert all(p in CLASS_NAMES for p in _b["predictions"])
print("Task 2D PASSED (3/3 pts)")

---
# Part 3 — Building APIs for Model Serving (CLO3) — 25 Points

## Demo — FastAPI Pattern

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field, validator
from typing import List

_demo = FastAPI(title="Demo")
class _DemoReq(BaseModel):
    value: float
@_demo.get("/health")
def _health(): return {"status": "ok"}
@_demo.post("/double")
def _double(req: _DemoReq): return {"result": req.value * 2}
_tc = TestClient(_demo)
print("Health:", _tc.get("/health").json())
print("Double:", _tc.post("/double", json={"value": 5.0}).json())

---
## Task 3A — Complete the Serving App (10 points)

In [ ]:
class PredictRequest(BaseModel):
    features: List[float]

    @validator("features")
    def check_length(cls, v):
        if len(v) != 6:
            raise ValueError("features must have exactly 6 elements")
        return v


class BatchPredictRequest(BaseModel):
    records: List[List[float]]

    @validator("records")
    def check_not_empty(cls, v):
        if len(v) == 0:
            raise ValueError("records must not be empty")
        return v


serving_app = FastAPI(title="Patient Risk Classifier", version="1.0.0")

MODEL_REGISTRY = {
    "model":     production_model,
    "metadata":  loaded_metadata,
    "version":   "v1.0.0",
    "loaded_at": datetime.utcnow().isoformat(),
}


@serving_app.get("/health")
def health():
    return {
        "status":        "ok",
        "model_version": MODEL_REGISTRY["version"],
        "loaded_at":     MODEL_REGISTRY["loaded_at"],
    }


@serving_app.post("/predict")
def predict(req: PredictRequest):
    features = np.array([req.features])
    proba    = MODEL_REGISTRY["model"].predict_proba(features)[0]
    class_id = int(np.argmax(proba))
    return {
        "prediction":   CLASS_NAMES[class_id],
        "confidence":   round(float(proba.max()), 4),
        "model_version": MODEL_REGISTRY["version"],
    }


@serving_app.post("/predict/batch")
def predict_batch_endpoint(req: BatchPredictRequest):
    data     = np.array(req.records)
    probas   = MODEL_REGISTRY["model"].predict_proba(data)
    class_ids = np.argmax(probas, axis=1)
    return {
        "predictions":   [CLASS_NAMES[c] for c in class_ids],
        "confidences":   [round(float(p.max()), 4) for p in probas],
        "count":         len(data),
        "model_version": MODEL_REGISTRY["version"],
    }


_c = TestClient(serving_app)
print(_c.get("/health").json())

---
## Task 3B — Test All Endpoints (5 points)

In [ ]:
_client = TestClient(serving_app)

health_response     = _client.get("/health").json()
predict_response    = _client.post("/predict", json={"features": X_test[0].tolist()}).json()
invalid_status_code = _client.post("/predict", json={"features": [1.0, 2.0, 3.0]}).status_code
batch_response      = _client.post("/predict/batch", json={"records": X_test[:5].tolist()}).json()
empty_batch_status  = _client.post("/predict/batch", json={"records": []}).status_code

print("Health       :", health_response)
print("Predict      :", predict_response)
print("Invalid code :", invalid_status_code)
print("Batch count  :", batch_response.get("count"))
print("Empty batch  :", empty_batch_status)

In [ ]:
from fastapi import FastAPI as _FA
assert isinstance(serving_app, _FA), "serving_app must be a FastAPI instance"
assert "features" in PredictRequest.__fields__
assert "records"  in BatchPredictRequest.__fields__
assert health_response and health_response.get("status") == "ok"
assert "model_version" in health_response
assert predict_response and predict_response["prediction"] in CLASS_NAMES
assert "confidence" in predict_response and "model_version" in predict_response
assert invalid_status_code in (400,422), f"Expected 422 or 400, got {invalid_status_code}"
assert batch_response and batch_response.get("count") == 5
assert len(batch_response["predictions"]) == 5
assert empty_batch_status in (400,422)
print("Part 3A+B PASSED — FastAPI serving app built and tested (15/15 pts)")

---
## Task 3C — Flask API Serving (CLO3) — 10 points

In [ ]:
import sys as _sys

FLASK_APP_CODE = """
import joblib, numpy as np
from flask import Flask, request, jsonify

app = Flask(__name__)

MODEL_PATH  = "/tmp/aiat125_final/model_bundle.joblib"
bundle      = joblib.load(MODEL_PATH)
model       = bundle["model"]
CLASS_NAMES = ["low_risk", "high_risk"]

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok"}), 200

@app.route("/predict", methods=["POST"])
def predict():
    data = request.get_json(silent=True)

    if data is None or "features" not in data:
        return jsonify({"error": "missing features"}), 400
    if len(data["features"]) != 6:
        return jsonify({"error": "features must have exactly 6 elements"}), 400
    try:
        for f in data["features"]:
            float(f)
    except (TypeError, ValueError):
        return jsonify({"error": "all features must be numeric"}), 400

    features  = np.array([data["features"]])
    class_id  = int(model.predict(features)[0])
    conf      = round(float(model.predict_proba(features)[0].max()), 3)
    return jsonify({"prediction": CLASS_NAMES[class_id], "class_id": class_id, "confidence": conf}), 200

if __name__ == "__main__":
    app.run(port=5000)
"""

if FLASK_APP_CODE:
    with open("/tmp/flask_serving.py", "w") as _f:
        _f.write(FLASK_APP_CODE)
    if "flask_serving" in _sys.modules:
        del _sys.modules["flask_serving"]
    _sys.path.insert(0, "/tmp")
    import flask_serving as _fm
    flask_client = _fm.app.test_client()
else:
    flask_client = None

flask_health_status  = flask_client.get("/health").status_code
flask_predict_result = flask_client.post("/predict", json={"features": X_test[0].tolist()}).get_json()
flask_invalid_status = flask_client.post("/predict", json={"features": [1, 2, 3]}).status_code

Q_FLASK_400_WHEN   = "Return 400 when the error is caused by the client's request — missing required fields, wrong data types, or invalid values."
Q_FLASK_500_WHEN   = "Return 500 when an unexpected server-side error occurs, such as a model loading failure or an unhandled exception during inference."
Q_FLASK_GUNICORN   = "Gunicorn is a production WSGI server that spawns multiple worker processes to handle concurrent requests; Flask's built-in dev server is single-threaded and not safe for production."
Q_FLASK_VS_FASTAPI = "Choose Flask for simpler APIs or when working with existing Flask codebases; FastAPI is better when you need automatic input validation, async support, and auto-generated API docs."

print("Flask health :", flask_health_status)
print("Flask predict:", flask_predict_result)
print("Flask invalid:", flask_invalid_status)

In [ ]:
assert FLASK_APP_CODE and isinstance(FLASK_APP_CODE, str)
assert flask_client is not None
assert flask_health_status == 200
assert flask_predict_result is not None
assert "prediction" in flask_predict_result
assert flask_predict_result["prediction"] in CLASS_NAMES
assert "confidence" in flask_predict_result
assert "class_id" in flask_predict_result
assert flask_invalid_status == 400, f"Invalid input must return 400, got {flask_invalid_status}"
assert Q_FLASK_400_WHEN and len(Q_FLASK_400_WHEN) > 10
assert Q_FLASK_500_WHEN and len(Q_FLASK_500_WHEN) > 10
assert Q_FLASK_GUNICORN and len(Q_FLASK_GUNICORN) > 10
assert Q_FLASK_VS_FASTAPI and len(Q_FLASK_VS_FASTAPI) > 10
print("Task 3C PASSED — Flask API (10/10 pts)")

---
# Part 4 — Cloud Deployment (CLO4) — 10 Points

## Task 4A — Cloud Platforms Knowledge (5 points)

In [ ]:
CLOUD_SERVICES = {
    "SageMaker":       "AWS",
    "Vertex AI":       "GCP",
    "Azure ML":        "Azure",
    "Lambda":          "AWS",
    "Cloud Run":       "GCP",
    "Azure Functions": "Azure",
    "EC2":             "AWS",
    "AKS":             "Azure",
}

INFERENCE_TYPE = {
    "Predict fraud on every credit card transaction as it happens": "real-time",
    "Score 1 million loan applications overnight":                  "batch",
    "Detect intrusion in a live network stream":                    "real-time",
    "Generate monthly customer churn predictions from CRM data":    "batch",
    "Return product recommendations while user browses the site":   "real-time",
}

CLOUD_SECURITY_PRACTICES = [
    "Use IAM roles with least-privilege access instead of storing credentials in code or environment variables.",
    "Enable encryption at rest and in transit for all model artifacts, data, and API communications.",
    "Authenticate all prediction endpoint requests using API keys, OAuth tokens, or mutual TLS.",
]

print("Cloud services mapping:", CLOUD_SERVICES)

In [ ]:
assert CLOUD_SERVICES["SageMaker"]       == "AWS"
assert CLOUD_SERVICES["Vertex AI"]       == "GCP"
assert CLOUD_SERVICES["Azure ML"]        == "Azure"
assert CLOUD_SERVICES["Lambda"]          == "AWS"
assert CLOUD_SERVICES["Cloud Run"]       == "GCP"
assert CLOUD_SERVICES["Azure Functions"] == "Azure"
assert CLOUD_SERVICES["EC2"]             == "AWS"
assert CLOUD_SERVICES["AKS"]             == "Azure"
for v in INFERENCE_TYPE.values():
    assert v in ("real-time","batch"), f"Invalid value: {v}"
assert INFERENCE_TYPE["Predict fraud on every credit card transaction as it happens"] == "real-time"
assert INFERENCE_TYPE["Score 1 million loan applications overnight"] == "batch"
assert INFERENCE_TYPE["Return product recommendations while user browses the site"] == "real-time"
assert len([p for p in CLOUD_SECURITY_PRACTICES if p and len(p)>5]) >= 3
print("Task 4A PASSED (5/5 pts)")

---
## Task 4B — Deployment Config + Authentication + Logging (5 points)

In [ ]:
cloud_deployment_config = {
    "model_name":            "patient-risk-classifier",
    "model_version":         "v1.0.0",
    "endpoint_type":         "real-time",
    "instance_type":         "ml.m5.large",
    "min_instances":         1,
    "max_instances":         10,
    "autoscaling_enabled":   True,
    "health_check_path":     "/health",
    "environment_variables": {"LOG_LEVEL": "INFO", "MODEL_VERSION": "v1.0.0"},
    "tags":                  {"team": "mlops", "project": "patient-risk"},
}


def api_key_auth(provided_key, valid_keys_set):
    if provided_key is None:
        return False
    return provided_key in valid_keys_set


def log_request(endpoint, status_code, latency_ms):
    return {
        "timestamp":   datetime.utcnow().isoformat(),
        "endpoint":    endpoint,
        "status_code": status_code,
        "latency_ms":  latency_ms,
    }


VALID_KEYS = {"key-abc-123", "key-xyz-456"}
print(api_key_auth("key-abc-123", VALID_KEYS))
print(api_key_auth("wrong", VALID_KEYS))
print(log_request("/predict", 200, 8.5))

In [ ]:
assert isinstance(cloud_deployment_config, dict)
for k in ["model_name","model_version","endpoint_type","instance_type",
          "min_instances","max_instances","autoscaling_enabled",
          "health_check_path","environment_variables","tags"]:
    assert k in cloud_deployment_config, f"missing '{k}'"
assert cloud_deployment_config["endpoint_type"] in ("real-time","batch","serverless")
assert isinstance(cloud_deployment_config["autoscaling_enabled"], bool)
assert cloud_deployment_config["min_instances"] <= cloud_deployment_config["max_instances"]
assert "team" in cloud_deployment_config["tags"] and "project" in cloud_deployment_config["tags"]
assert api_key_auth("key-abc-123", VALID_KEYS) is True
assert api_key_auth("wrong", VALID_KEYS) is False
assert api_key_auth(None, VALID_KEYS) is False
_l = log_request("/predict", 200, 8.5)
for k in ["timestamp","endpoint","status_code","latency_ms"]: assert k in _l
assert _l["endpoint"] == "/predict" and _l["status_code"] == 200
print("Part 4 PASSED (10/10 pts)")

---
# Part 5 — Containers & Orchestration (CLO5) — 15 Points

## Task 5A — Write a Dockerfile (4 points)

In [ ]:
DOCKERFILE_CONTENT = """FROM python:3.9-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install -r requirements.txt
COPY . .
EXPOSE 8000
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
"""

with open(os.path.join(BASE_DIR, "Dockerfile"), "w") as f:
    f.write(DOCKERFILE_CONTENT)
print(DOCKERFILE_CONTENT)

---
## Task 5B — Kubernetes: Deployment + Service + HPA (7 points)

In [ ]:
K8S_DEPLOYMENT_YAML = """apiVersion: apps/v1
kind: Deployment
metadata:
  name: patient-risk-classifier
  labels:
    app: patient-risk-classifier
spec:
  replicas: 3
  selector:
    matchLabels:
      app: patient-risk-classifier
  strategy:
    type: RollingUpdate
    rollingUpdate:
      maxSurge: 1
      maxUnavailable: 0
  template:
    metadata:
      labels:
        app: patient-risk-classifier
    spec:
      containers:
        - name: patient-risk-classifier
          image: myregistry/patient-risk-classifier:v1.0.0
          ports:
            - containerPort: 8000
          resources:
            requests:
              cpu: "250m"
              memory: "256Mi"
            limits:
              cpu: "500m"
              memory: "512Mi"
          readinessProbe:
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 10
            periodSeconds: 5
          livenessProbe:
            httpGet:
              path: /health
              port: 8000
            initialDelaySeconds: 30
            periodSeconds: 10
"""

K8S_SERVICE_YAML = """apiVersion: v1
kind: Service
metadata:
  name: patient-risk-classifier-svc
spec:
  type: LoadBalancer
  selector:
    app: patient-risk-classifier
  ports:
    - name: http
      port: 80
      targetPort: 8000
      protocol: TCP
"""

K8S_HPA_YAML = """apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: patient-risk-classifier-hpa
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: patient-risk-classifier
  minReplicas: 2
  maxReplicas: 10
  metrics:
    - type: Resource
      resource:
        name: cpu
        target:
          type: Utilization
          averageUtilization: 70
"""

Q_K8S_ROLLING = "A rolling update gradually replaces old pods with new ones one at a time, ensuring the required number of ready pods always serve traffic so users experience zero downtime."
Q_K8S_HPA     = "The HPA watches average CPU utilization across all pods; when it exceeds 70%, it adds pods up to maxReplicas, and when it drops, it removes pods down to minReplicas."

for name, content in [("deployment.yaml", K8S_DEPLOYMENT_YAML),
                       ("service.yaml",    K8S_SERVICE_YAML),
                       ("hpa.yaml",        K8S_HPA_YAML)]:
    with open(os.path.join(BASE_DIR, name), "w") as f:
        f.write(content)
    print(f"--- {name} ---")
    print(content)

---
## Task 5C — CI/CD Pipeline Knowledge (4 points)

In [ ]:
CICD_PIPELINE_STAGES = [
    {"name": "Code Checkout & Lint",   "description": "Pull source code from version control and run linting and static analysis checks."},
    {"name": "Unit Tests",             "description": "Run automated unit tests to verify each component works correctly in isolation."},
    {"name": "Model Train & Evaluate", "description": "Train the model on the latest data and evaluate accuracy against the acceptance threshold."},
    {"name": "Docker Build & Push",    "description": "Build the Docker image with the trained model artifact and push it to the container registry."},
    {"name": "Deploy to Production",   "description": "Apply updated Kubernetes manifests to deploy the new model version to the production cluster."},
]

Q5_ROLLBACK  = "A rollback strategy reverts to the previous known-good model version when the new deployment fails validation or causes errors in production."
Q5_CANARY    = "Canary deployment routes a small percentage of traffic to the new model first; if metrics stay healthy the percentage is gradually increased until full rollout."
Q5_BLUEGREEN = "Blue-green deployment maintains two identical environments; after deploying to the inactive one, traffic is switched atomically, enabling instant rollback by switching back."

for i, s in enumerate(CICD_PIPELINE_STAGES, 1):
    print(f"  {i}. {s['name']}: {s['description']}")

In [ ]:
assert DOCKERFILE_CONTENT and isinstance(DOCKERFILE_CONTENT, str)
df = DOCKERFILE_CONTENT.upper()
for kw in ["FROM","WORKDIR","COPY","RUN","EXPOSE","CMD"]: assert kw in df, f"Dockerfile missing {kw}"
assert "8000" in DOCKERFILE_CONTENT
assert "uvicorn" in DOCKERFILE_CONTENT.lower() or "gunicorn" in DOCKERFILE_CONTENT.lower()
print("5A: Dockerfile OK (4/4 pts)")

assert K8S_DEPLOYMENT_YAML and isinstance(K8S_DEPLOYMENT_YAML, str)
ky = K8S_DEPLOYMENT_YAML
assert "apiVersion" in ky and "apps/v1" in ky
assert "kind: Deployment" in ky
assert "patient-risk-classifier" in ky
assert "replicas: 3" in ky or "replicas:3" in ky.replace(" ","")
assert "8000" in ky
assert "resources" in ky and "cpu" in ky and "memory" in ky
assert "livenessProbe" in ky or "readinessProbe" in ky
print("5B-i: Deployment YAML OK")

assert K8S_SERVICE_YAML and isinstance(K8S_SERVICE_YAML, str)
svc = K8S_SERVICE_YAML
assert "kind: Service" in svc
assert "LoadBalancer" in svc
assert "patient-risk-classifier" in svc
assert "8000" in svc or "targetPort" in svc
print("5B-ii: Service YAML OK")

assert K8S_HPA_YAML and isinstance(K8S_HPA_YAML, str)
hpa = K8S_HPA_YAML
assert "HorizontalPodAutoscaler" in hpa
assert "minReplicas" in hpa and "maxReplicas" in hpa
assert "70" in hpa
assert "patient-risk-classifier" in hpa
print("5B: Kubernetes YAMLs OK (7/7 pts)")

assert len(CICD_PIPELINE_STAGES) == 5
for i, s in enumerate(CICD_PIPELINE_STAGES):
    assert s.get("name") and len(s["name"]) > 2
    assert s.get("description") and len(s["description"]) > 5
assert Q5_ROLLBACK and Q5_CANARY and Q5_BLUEGREEN
assert all(len(q) > 10 for q in [Q5_ROLLBACK, Q5_CANARY, Q5_BLUEGREEN])
print("5C: CI/CD OK (4/4 pts)")

print("\nPart 5 PASSED — Docker + Kubernetes + CI/CD (15/15 pts)")

---
# Part 6 — Monitoring, Maintenance & MLOps (CLO6) — 25 Points

## Task 6A — Model Performance Monitor (3 points)

In [ ]:
class ModelMonitor:
    """Tracks model performance in a production environment."""

    def __init__(self, model_name: str, accuracy_threshold: float = 0.80):
        self.model_name         = model_name
        self.accuracy_threshold = accuracy_threshold
        self.request_count      = 0
        self.correct_count      = 0
        self.latencies_ms       = []
        self.alerts             = []

    def log_prediction(self, predicted: str, actual: str, latency_ms: float):
        self.request_count += 1
        if predicted == actual:
            self.correct_count += 1
        self.latencies_ms.append(latency_ms)
        if self.request_count >= 10 and self.current_accuracy() < self.accuracy_threshold:
            self.alerts.append(
                f"Accuracy {self.current_accuracy():.3f} below threshold {self.accuracy_threshold}"
            )

    def current_accuracy(self) -> float:
        return self.correct_count / self.request_count if self.request_count > 0 else 0.0

    def p95_latency(self) -> float:
        return float(np.percentile(self.latencies_ms, 95)) if self.latencies_ms else 0.0

    def summary(self) -> dict:
        return {
            "model_name":     self.model_name,
            "request_count":  self.request_count,
            "accuracy":       self.current_accuracy(),
            "p95_latency_ms": self.p95_latency(),
            "alert_count":    len(self.alerts),
        }


monitor = ModelMonitor("patient-risk-v1", accuracy_threshold=0.80)
for pred, true in zip(
    [CLASS_NAMES[p] for p in production_model.predict(X_test[:20])],
    [CLASS_NAMES[t] for t in y_test[:20]]
):
    monitor.log_prediction(pred, true, latency_ms=np.random.uniform(1, 20))

print("Monitor summary:", monitor.summary())

In [ ]:
assert isinstance(monitor, ModelMonitor)
assert monitor.request_count == 20
assert 0 <= monitor.current_accuracy() <= 1
assert monitor.p95_latency() > 0
_s = monitor.summary()
for k in ["model_name","request_count","accuracy","p95_latency_ms","alert_count"]: assert k in _s
assert _s["request_count"] == 20
print("Task 6A PASSED (3/3 pts)")

---
## Task 6B — Data Drift Detection (3 points)

In [ ]:
np.random.seed(99)
drifted_data = X_test.copy()
drifted_data[:, 0] += 2.0
drifted_data[:, 1] -= 1.5


def detect_drift(reference, production, feature_names, alpha=0.05):
    drifted_features = []
    feature_results  = {}
    for i, name in enumerate(feature_names):
        stat, p_value = stats.ks_2samp(reference[:, i], production[:, i])
        drifted = p_value < alpha
        if drifted:
            drifted_features.append(name)
        feature_results[name] = {"p_value": round(float(p_value), 6), "drifted": drifted}
    return {
        "drifted_features": drifted_features,
        "feature_results":  feature_results,
        "overall_drift":    len(drifted_features) > 0,
    }


def should_retrain(drift_result, max_drifted_features=2):
    return len(drift_result["drifted_features"]) >= max_drifted_features


drift_result   = detect_drift(X_train, drifted_data, FEATURE_NAMES)
retrain_needed = should_retrain(drift_result)

print("Drifted:", drift_result["drifted_features"])
print("Retrain:", retrain_needed)

In [ ]:
assert callable(detect_drift) and callable(should_retrain)
assert isinstance(drift_result, dict)
for k in ["drifted_features","feature_results","overall_drift"]: assert k in drift_result
assert "age_norm" in drift_result["drifted_features"]
assert "bp_norm"  in drift_result["drifted_features"]
assert drift_result["overall_drift"] is True
assert retrain_needed is True
_nd = detect_drift(X_train, X_test, FEATURE_NAMES)
assert len(_nd["drifted_features"]) < len(FEATURE_NAMES)
print("Task 6B PASSED (3/3 pts)")

---
## Task 6C — MLflow Experiment Tracking (3 points)

In [ ]:
import mlflow, mlflow.sklearn

mlflow.set_tracking_uri(f"file://{BASE_DIR}mlruns")
mlflow.set_experiment("patient-risk-classifier")

retrained_model  = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
retrained_model.fit(X_train, y_train)
retrain_accuracy = accuracy_score(y_test, retrained_model.predict(X_test))

mlflow_run_id = None
with mlflow.start_run(run_name="retraining-run-v2") as run:
    mlflow.log_params({"n_estimators": 200, "max_depth": 10, "random_state": 42})
    mlflow.log_metrics({"accuracy": retrain_accuracy, "training_samples": len(X_train)})
    mlflow.set_tag("model_type", "RandomForest")
    mlflow.sklearn.log_model(retrained_model, artifact_path="model")
    mlflow_run_id = run.info.run_id

RETRAINING_STRATEGY = {
    "trigger_conditions": [
        "Accuracy drops below 0.80 on the monitoring window",
        "Drift detected in 2 or more input features",
        "Monthly scheduled retraining regardless of drift",
    ],
    "retraining_frequency":    "weekly",
    "validation_threshold":     0.80,
    "rollback_on_degradation": True,
}

TRACKING_TOOL_COMPARISON = {
    "mlflow_best_for":  "Self-hosted on-premise deployments where data privacy prevents sending experiment data to external cloud services.",
    "wandb_best_for":   "Teams wanting a rich collaborative dashboard, easy hyperparameter sweeps, and a managed cloud-hosted experiment tracking service.",
    "key_difference":   "MLflow is open-source and runs locally, while W&B is a managed cloud service with a richer collaboration UI.",
}

print(f"Retrained accuracy: {retrain_accuracy:.4f}")
print(f"MLflow run ID     : {mlflow_run_id}")

In [ ]:
assert mlflow_run_id and isinstance(mlflow_run_id, str) and len(mlflow_run_id) > 5
assert isinstance(RETRAINING_STRATEGY, dict)
for k in ["trigger_conditions","retraining_frequency","validation_threshold","rollback_on_degradation"]:
    assert k in RETRAINING_STRATEGY, f"missing '{k}'"
assert len(RETRAINING_STRATEGY["trigger_conditions"]) >= 3
assert 0 < RETRAINING_STRATEGY["validation_threshold"] <= 1
assert isinstance(RETRAINING_STRATEGY["rollback_on_degradation"], bool)
assert isinstance(TRACKING_TOOL_COMPARISON, dict)
for k in ["mlflow_best_for","wandb_best_for","key_difference"]:
    assert k in TRACKING_TOOL_COMPARISON and len(TRACKING_TOOL_COMPARISON[k]) > 10
print("Task 6C PASSED (3/3 pts)")

---
## Task 6D — Model Versioning Registry (3 points)

In [ ]:
class ModelVersionRegistry:
    """A simple registry for managing model versions."""

    def __init__(self):
        self.versions           = {}
        self.production_version = None
        self._history           = []

    def register(self, model, version: str, metrics: dict):
        if version in self.versions:
            raise ValueError(f"Version '{version}' already exists")
        self.versions[version] = {
            "model":         model,
            "metrics":       metrics,
            "registered_at": datetime.utcnow().isoformat(),
            "stage":         "staging",
        }

    def promote(self, version: str):
        if version not in self.versions:
            raise ValueError(f"Version '{version}' not found in registry")
        self.versions[version]["stage"] = "production"
        self.production_version = version
        self._history.append(version)

    def rollback(self):
        if self.production_version is None:
            return
        self.versions[self.production_version]["stage"] = "archived"
        self._history.pop()
        if self._history:
            prev = self._history[-1]
            self.versions[prev]["stage"] = "production"
            self.production_version = prev
        else:
            self.production_version = None

    def get_production_model(self):
        if self.production_version is None:
            return None
        return self.versions[self.production_version]["model"]

    def list_versions(self) -> dict:
        return {
            v: {
                "stage":         info["stage"],
                "metrics":       info["metrics"],
                "registered_at": info["registered_at"],
            }
            for v, info in self.versions.items()
        }


registry = ModelVersionRegistry()
registry.register(production_model, "v1.0.0", {"accuracy": test_accuracy})
registry.register(retrained_model,  "v2.0.0", {"accuracy": retrain_accuracy})
registry.promote("v1.0.0")
print("Production after promote v1:", registry.production_version)
registry.promote("v2.0.0")
print("Production after promote v2:", registry.production_version)
registry.rollback()
print("Production after rollback   :", registry.production_version)
print("Versions:", registry.list_versions())

In [ ]:
assert isinstance(registry, ModelVersionRegistry)
assert len(registry.versions) == 2
try: registry.register(production_model, "v1.0.0", {}); assert False
except ValueError: pass
try: registry.promote("v99.0.0"); assert False
except ValueError: pass
assert registry.production_version == "v1.0.0", \
    f"After rollback should revert to v1.0.0, got: {registry.production_version}"
_pm = registry.get_production_model()
assert _pm is not None
_lv = registry.list_versions()
assert isinstance(_lv, dict) and len(_lv) == 2
for v in _lv.values():
    assert "stage" in v and "metrics" in v
print("Task 6D PASSED — model versioning registry (3/3 pts)")

---
## Task 6E — Automated Retraining Pipeline (3 points)

In [ ]:
def retraining_pipeline(X_train, y_train, X_test, y_test,
                        drift_result, current_acc, registry,
                        new_version, threshold=0.80):
    if not should_retrain(drift_result):
        return {"retrained": False, "reason": "no_drift"}

    new_model = RandomForestClassifier(n_estimators=200, random_state=0)
    new_model.fit(X_train, y_train)
    new_acc = accuracy_score(y_test, new_model.predict(X_test))

    registry.register(new_model, new_version, {"accuracy": new_acc})

    if new_acc >= threshold and new_acc >= current_acc:
        registry.promote(new_version)
        return {
            "retrained":    True,
            "promoted":     True,
            "new_accuracy": new_acc,
            "version":      new_version,
        }
    else:
        return {
            "retrained":    True,
            "promoted":     False,
            "new_accuracy": new_acc,
            "reason":       "accuracy_degraded",
        }


registry2 = ModelVersionRegistry()
registry2.register(production_model, "v1.0.0", {"accuracy": test_accuracy})
registry2.promote("v1.0.0")

pipeline_result = retraining_pipeline(
    X_train, y_train, X_test, y_test,
    drift_result=drift_result,
    current_acc=test_accuracy,
    registry=registry2,
    new_version="v2.0.0"
)
print("Pipeline result (drift)    :", pipeline_result)

no_drift_result = detect_drift(X_train, X_test, FEATURE_NAMES)
pipeline_result_no_drift = retraining_pipeline(
    X_train, y_train, X_test, y_test,
    drift_result=no_drift_result,
    current_acc=test_accuracy,
    registry=registry2,
    new_version="v3.0.0"
)
print("Pipeline result (no drift) :", pipeline_result_no_drift)

In [ ]:
assert callable(retraining_pipeline)
assert isinstance(pipeline_result, dict)
assert "retrained" in pipeline_result
assert pipeline_result["retrained"] is True
assert "new_accuracy" in pipeline_result
assert 0 < pipeline_result["new_accuracy"] <= 1
assert pipeline_result_no_drift["retrained"] is False
assert pipeline_result_no_drift.get("reason") == "no_drift"
print("Task 6E PASSED — retraining pipeline (3/3 pts)")

---
## Task 6F — A/B Testing (4 points)

In [ ]:
class ABTestRouter:
    """Routes requests between two models and tracks performance."""

    def __init__(self, model_a, model_b, traffic_to_b: float = 0.5):
        self.model_a      = model_a
        self.model_b      = model_b
        self.traffic_to_b = traffic_to_b
        self.results_a    = []
        self.results_b    = []

    def predict(self, features: list, true_label=None):
        use_b      = np.random.random() < self.traffic_to_b
        model      = self.model_b if use_b else self.model_a
        model_name = "B" if use_b else "A"
        class_index = int(model.predict(np.array([features]))[0])
        prediction  = CLASS_NAMES[class_index]
        if true_label is not None:
            correct = int(class_index == true_label)
            (self.results_b if use_b else self.results_a).append(correct)
        return {"prediction": prediction, "model": model_name}

    def accuracy_a(self) -> float:
        return float(np.mean(self.results_a)) if self.results_a else 0.0

    def accuracy_b(self) -> float:
        return float(np.mean(self.results_b)) if self.results_b else 0.0

    def winner(self) -> str:
        if not self.results_a or not self.results_b:
            return "insufficient_data"
        return "B" if self.accuracy_b() > self.accuracy_a() else "A"

    def summary(self) -> dict:
        return {
            "requests_a": len(self.results_a),
            "requests_b": len(self.results_b),
            "accuracy_a": self.accuracy_a(),
            "accuracy_b": self.accuracy_b(),
            "winner":     self.winner(),
        }


np.random.seed(7)
ab_router = ABTestRouter(production_model, retrained_model, traffic_to_b=0.5)
for i in range(200):
    ab_router.predict(
        features=X_test[i % len(X_test)].tolist(),
        true_label=y_test[i % len(y_test)]
    )

ab_summary = ab_router.summary()
print("A/B Summary:", ab_summary)

In [ ]:
assert isinstance(ab_router, ABTestRouter)
assert ab_router.results_a and ab_router.results_b
total = len(ab_router.results_a) + len(ab_router.results_b)
assert total == 200, f"Total requests must be 200, got {total}"
assert 0 <= ab_router.accuracy_a() <= 1
assert 0 <= ab_router.accuracy_b() <= 1
assert ab_router.winner() in ("A","B")
assert isinstance(ab_summary, dict)
for k in ["requests_a","requests_b","accuracy_a","accuracy_b","winner"]:
    assert k in ab_summary
assert ab_summary["requests_a"] + ab_summary["requests_b"] == 200
print(f"Task 6F PASSED — A={ab_summary['accuracy_a']:.3f} vs B={ab_summary['accuracy_b']:.3f}, winner={ab_summary['winner']} (4/4 pts)")

---
## Task 6G — Canary Deployment (3 points)

In [ ]:
class CanaryDeployment:
    """Manages gradual rollout of a new model."""

    def __init__(self, current_model, new_model, canary_fraction: float = 0.10):
        self.current_model   = current_model
        self.new_model       = new_model
        self.canary_fraction = canary_fraction
        self.current_metrics = []
        self.canary_metrics  = []
        self.status          = "running"

    def route_request(self, features: list):
        use_canary  = np.random.random() < self.canary_fraction
        model       = self.new_model if use_canary else self.current_model
        model_name  = "canary" if use_canary else "current"
        class_index = int(model.predict(np.array([features]))[0])
        prediction  = CLASS_NAMES[class_index]
        return {"prediction": prediction, "model": model_name}

    def log_outcome(self, model_name: str, correct: int):
        if model_name == "canary":
            self.canary_metrics.append(correct)
        else:
            self.current_metrics.append(correct)

    def canary_accuracy(self) -> float:
        return float(np.mean(self.canary_metrics)) if self.canary_metrics else 0.0

    def current_accuracy(self) -> float:
        return float(np.mean(self.current_metrics)) if self.current_metrics else 0.0

    def increase_traffic(self, new_fraction: float):
        if new_fraction > 1.0:
            raise ValueError("new_fraction cannot exceed 1.0")
        self.canary_fraction = new_fraction

    def promote(self):
        self.canary_fraction = 1.0
        self.status = "promoted"

    def rollback(self):
        self.canary_fraction = 0.0
        self.status = "rolled_back"

    def summary(self) -> dict:
        return {
            "status":           self.status,
            "canary_fraction":  self.canary_fraction,
            "canary_accuracy":  self.canary_accuracy(),
            "current_accuracy": self.current_accuracy(),
            "canary_requests":  len(self.canary_metrics),
            "current_requests": len(self.current_metrics),
        }


np.random.seed(42)
canary = CanaryDeployment(production_model, retrained_model, canary_fraction=0.10)

for i in range(100):
    feat = X_test[i % len(X_test)].tolist()
    res  = canary.route_request(feat)
    true_cls = y_test[i % len(y_test)]
    pred_cls = production_model.predict(np.array([feat]))[0] if res["model"] == "current" else retrained_model.predict(np.array([feat]))[0]
    canary.log_outcome(res["model"], int(pred_cls == true_cls))

print("Phase 1 (10%):", canary.summary())

canary.increase_traffic(0.50)
for i in range(100, 200):
    feat = X_test[i % len(X_test)].tolist()
    res  = canary.route_request(feat)
    true_cls = y_test[i % len(y_test)]
    pred_cls = production_model.predict(np.array([feat]))[0] if res["model"] == "current" else retrained_model.predict(np.array([feat]))[0]
    canary.log_outcome(res["model"], int(pred_cls == true_cls))

print("Phase 2 (50%):", canary.summary())

if canary.canary_accuracy() >= canary.current_accuracy():
    canary.promote()
else:
    canary.rollback()

print("Final status  :", canary.summary())

In [ ]:
assert isinstance(canary, CanaryDeployment)
assert canary.status in ("promoted","rolled_back")
if canary.status == "promoted":
    assert canary.canary_fraction == 1.0
else:
    assert canary.canary_fraction == 0.0
try: canary.increase_traffic(1.5); assert False
except ValueError: pass
_cs = canary.summary()
for k in ["status","canary_fraction","canary_accuracy","current_accuracy",
          "canary_requests","current_requests"]:
    assert k in _cs
assert _cs["canary_requests"] + _cs["current_requests"] == 200
print(f"Task 6G PASSED — status={_cs['status']}, canary_acc={_cs['canary_accuracy']:.3f} (3/3 pts)")

---
## Task 6H — Alerting & Incident Management (3 points)

In [ ]:
from datetime import datetime as _dt, timedelta as _td

class AlertManager:
    """Checks production request logs against alert thresholds."""

    def check_error_rate(self, logs: list, threshold: float = 0.01):
        if not logs:
            return None
        n_errors = sum(1 for r in logs if r.get("error", False))
        rate = n_errors / len(logs)
        if rate > threshold:
            return (f"ALERT [error_rate]: {rate:.2%} > threshold {threshold:.2%} "
                    f"({n_errors}/{len(logs)} errors)")
        return None

    def check_latency(self, logs: list, p99_threshold_ms: float = 500.0):
        latencies = [r["latency_ms"] for r in logs if not r.get("error", False)]
        if not latencies:
            return None
        p99 = float(np.percentile(latencies, 99))
        if p99 > p99_threshold_ms:
            return f"ALERT [latency_p99]: {p99:.1f}ms > threshold {p99_threshold_ms:.1f}ms"
        return None

    def check_confidence(self, logs: list, threshold: float = 0.6):
        confidences = [r["confidence"] for r in logs
                       if not r.get("error", False) and "confidence" in r]
        if not confidences:
            return None
        mean_conf = float(np.mean(confidences))
        if mean_conf < threshold:
            return f"ALERT [confidence]: mean={mean_conf:.3f} < threshold {threshold:.3f}"
        return None

    def check_service_alive(self, logs: list, window_minutes: float = 5.0):
        cutoff = _dt.utcnow() - _td(minutes=window_minutes)
        recent = [r for r in logs
                  if not r.get("error", False)
                  and _dt.fromisoformat(r["timestamp"]) > cutoff]
        if not recent:
            return f"ALERT [dead_service]: No successful predictions in last {window_minutes:.0f} minutes."
        return None

    def run_all_checks(self, logs: list) -> list:
        alerts = []
        for check_fn in [self.check_error_rate, self.check_latency,
                         self.check_confidence, self.check_service_alive]:
            result = check_fn(logs)
            if result:
                alerts.append(result)
        return alerts


_rng2 = np.random.default_rng(99)
_now2 = _dt.utcnow()
alert_logs = []
for i in range(200):
    _ts = _now2 - _td(minutes=10) + _td(seconds=i * 3)
    alert_logs.append({
        "timestamp":  _ts.isoformat(),
        "latency_ms": round(float(_rng2.normal(40, 10)), 2),
        "confidence": round(float(np.clip(_rng2.normal(0.85, 0.08), 0.5, 1.0)), 4),
        "error":      False,
    })

for _idx in _rng2.choice(200, size=15, replace=False):
    alert_logs[_idx]["error"]      = True
    alert_logs[_idx]["confidence"] = 0.0

for _idx in _rng2.choice(200, size=10, replace=False):
    if not alert_logs[_idx]["error"]:
        alert_logs[_idx]["latency_ms"] = float(_rng2.uniform(700, 1200))

alert_manager = AlertManager()
fired_alerts  = alert_manager.run_all_checks(alert_logs)

print(f"Alerts fired ({len(fired_alerts)}):")
for _a in fired_alerts:
    print(" ", _a)

In [ ]:
assert isinstance(alert_manager, AlertManager)
assert isinstance(fired_alerts, list)
assert alert_manager.check_error_rate(alert_logs) is not None
_clean = [{"timestamp": _dt.utcnow().isoformat(), "latency_ms": 30.0, "confidence": 0.9, "error": False}] * 50
assert alert_manager.check_error_rate(_clean) is None
assert alert_manager.check_confidence(_clean) is None
_old = [{"timestamp": (_dt.utcnow() - _td(minutes=20)).isoformat(),
          "latency_ms": 30.0, "confidence": 0.9, "error": False}]
assert alert_manager.check_service_alive(_old) is not None
assert isinstance(alert_manager.run_all_checks(_clean), list)
print("Task 6H PASSED — AlertManager (3/3 pts)")

---
# Final Deployment Gate — Score Summary

In [ ]:
RESULTS = {}
SCORED  = {}

def chk(label, pts, fn):
    try:
        fn()
        RESULTS[label] = "PASS"
        SCORED[label]  = pts
    except Exception as e:
        RESULTS[label] = f"FAIL — {e}"
        SCORED[label]  = 0

def _chk1a():
    assert sorted(LIFECYCLE_STAGES.values()) == [1,2,3,4,5,6]
    assert all(v and len(v) > 5 for v in STAGE_DESCRIPTIONS.values())
    assert FEEDBACK_LOOP_DEFINITION and len(FEEDBACK_LOOP_DEFINITION) > 10

def _chk1b():
    assert isinstance(production_model, RFC) and production_model.n_estimators == 100
    assert test_accuracy >= DEPLOY_THRESHOLD and deployment_ready
    for k in ["model_type","test_accuracy","threshold","ready","timestamp"]:
        assert k in deployment_report

def _chk2a():
    assert os.path.exists(PICKLE_PATH) and os.path.exists(JOBLIB_PATH) and os.path.exists(META_PATH)
    assert "model" not in json.loads(open(META_PATH).read())
    assert loaded_model is not None

def _chk2b():
    assert os.path.exists(ONNX_PATH) and os.path.getsize(ONNX_PATH) > 1000
    assert onnx_preds is not None and len(onnx_preds) == 5
    assert np.array_equal(np.array(onnx_preds).flatten()[:5], production_model.predict(X_test[:5]))

def _chk2c():
    assert TF_SERVING_CONFIG and "model_config" in TF_SERVING_CONFIG.lower()
    assert isinstance(TORCHSERVE_MANIFEST, dict)
    for k in ["model_name","handler","serialized_file","model_file","version"]:
        assert k in TORCHSERVE_MANIFEST

def _chk2d():
    r = predict_single(production_model, X_test[0].tolist())
    assert r["prediction"] in CLASS_NAMES
    b = predict_batch(production_model, X_test[:5])
    assert b["count"] == 5

def _chk3ab():
    from fastapi import FastAPI as _FA
    assert isinstance(serving_app, _FA)
    assert health_response and health_response.get("status") == "ok" and "model_version" in health_response
    assert predict_response and predict_response["prediction"] in CLASS_NAMES
    assert invalid_status_code in (400, 422)
    assert batch_response and batch_response.get("count") == 5
    assert empty_batch_status in (400, 422)

def _chk3c():
    assert FLASK_APP_CODE and isinstance(FLASK_APP_CODE, str)
    assert flask_client is not None
    assert flask_health_status == 200
    assert flask_predict_result and flask_predict_result.get("prediction") in CLASS_NAMES
    assert flask_invalid_status == 400
    assert Q_FLASK_400_WHEN and Q_FLASK_500_WHEN and Q_FLASK_GUNICORN and Q_FLASK_VS_FASTAPI

def _chk4a():
    assert CLOUD_SERVICES["SageMaker"] == "AWS" and CLOUD_SERVICES["Vertex AI"] == "GCP"
    assert all(v in ("real-time","batch") for v in INFERENCE_TYPE.values())

def _chk4b():
    for k in ["model_name","endpoint_type","min_instances","max_instances",
              "autoscaling_enabled","health_check_path","tags"]:
        assert k in cloud_deployment_config
    assert api_key_auth("key-abc-123", VALID_KEYS) is True
    assert api_key_auth(None, VALID_KEYS) is False
    _l = log_request("/p", 200, 1.0)
    for k in ["timestamp","endpoint","status_code","latency_ms"]: assert k in _l

def _chk5():
    df = (DOCKERFILE_CONTENT or "").upper()
    for kw in ["FROM","WORKDIR","COPY","RUN","EXPOSE","CMD"]: assert kw in df
    assert "8000" in (DOCKERFILE_CONTENT or "")
    ky = K8S_DEPLOYMENT_YAML or ""
    assert all(kw in ky for kw in ["apiVersion","Deployment","patient-risk-classifier","8000","resources"])
    assert K8S_SERVICE_YAML and "LoadBalancer" in K8S_SERVICE_YAML
    assert K8S_HPA_YAML and "HorizontalPodAutoscaler" in K8S_HPA_YAML and "70" in K8S_HPA_YAML
    assert len(CICD_PIPELINE_STAGES) == 5 and all(s.get("name") for s in CICD_PIPELINE_STAGES)

def _chk6a():
    assert monitor.request_count == 20
    _s = monitor.summary()
    for k in ["model_name","request_count","accuracy","p95_latency_ms","alert_count"]: assert k in _s

def _chk6b():
    assert "age_norm" in drift_result["drifted_features"]
    assert drift_result["overall_drift"] is True
    assert retrain_needed is True

def _chk6c():
    assert mlflow_run_id and len(mlflow_run_id) > 5
    for k in ["trigger_conditions","retraining_frequency","validation_threshold","rollback_on_degradation"]:
        assert k in RETRAINING_STRATEGY
    assert len(RETRAINING_STRATEGY["trigger_conditions"]) >= 3
    assert isinstance(TRACKING_TOOL_COMPARISON, dict)
    for k in ["mlflow_best_for","wandb_best_for","key_difference"]:
        assert k in TRACKING_TOOL_COMPARISON and len(TRACKING_TOOL_COMPARISON[k]) > 10

def _chk6d():
    assert len(registry.versions) == 2
    assert registry.production_version == "v1.0.0"
    assert registry.get_production_model() is not None

def _chk6e():
    assert pipeline_result["retrained"] is True
    assert pipeline_result_no_drift["retrained"] is False

def _chk6f():
    total = len(ab_router.results_a) + len(ab_router.results_b)
    assert total == 200
    assert ab_router.winner() in ("A","B")
    _s = ab_router.summary()
    for k in ["requests_a","requests_b","accuracy_a","accuracy_b","winner"]: assert k in _s

def _chk6g():
    assert canary.status in ("promoted","rolled_back")
    _cs = canary.summary()
    assert _cs["canary_requests"] + _cs["current_requests"] == 200

def _chk6h():
    assert isinstance(alert_manager, AlertManager)
    assert isinstance(fired_alerts, list)
    assert alert_manager.check_error_rate(alert_logs) is not None
    _clean = [{"timestamp": _dt.utcnow().isoformat(), "latency_ms": 30.0, "confidence": 0.9, "error": False}] * 50
    assert alert_manager.check_error_rate(_clean) is None
    _old = [{"timestamp": (_dt.utcnow() - _td(minutes=20)).isoformat(),
              "latency_ms": 30.0, "confidence": 0.9, "error": False}]
    assert alert_manager.check_service_alive(_old) is not None

chk("1A  Lifecycle Stages          (CLO1)",  5, _chk1a)
chk("1B  Train + Validate          (CLO1)",  5, _chk1b)
chk("2A  Pickle + Joblib           (CLO2)",  4, _chk2a)
chk("2B  ONNX                      (CLO2)",  5, _chk2b)
chk("2C  Serving Frameworks        (CLO2)",  3, _chk2c)
chk("2D  Batch + Single Predict    (CLO2)",  3, _chk2d)
chk("3AB FastAPI + Testing         (CLO3)", 15, _chk3ab)
chk("3C  Flask API                 (CLO3)", 10, _chk3c)
chk("4A  Cloud Platforms           (CLO4)",  5, _chk4a)
chk("4B  Config + Auth + Logging   (CLO4)",  5, _chk4b)
chk("5   Docker + K8s + CI/CD      (CLO5)", 15, _chk5)
chk("6A  Performance Monitor       (CLO6)",  3, _chk6a)
chk("6B  Drift Detection           (CLO6)",  3, _chk6b)
chk("6C  MLflow + WandB            (CLO6)",  3, _chk6c)
chk("6D  Model Versioning          (CLO6)",  3, _chk6d)
chk("6E  Retraining Pipeline       (CLO6)",  3, _chk6e)
chk("6F  A/B Testing               (CLO6)",  4, _chk6f)
chk("6G  Canary Deployment         (CLO6)",  3, _chk6g)
chk("6H  AlertManager              (CLO6)",  3, _chk6h)

total = sum(SCORED.values())
print("=" * 65)
print("  AIAT 125 — SOLUTION KEY: DEPLOYMENT GATE REPORT")
print("=" * 65)
for label, status in RESULTS.items():
    pts  = SCORED[label]
    icon = "PASS" if status == "PASS" else "FAIL"
    print(f"  [{icon}] {label}  (+{pts}pts)")
    if status != "PASS":
        print(f"         {status}")
print("-" * 65)
print(f"  Total: {total} / 100 pts")
if total == 100:
    print("  PERFECT SCORE — All CLOs achieved!")
print("=" * 65)

---
# Closing Reflection — For Class Discussion

1. **CLO1**: What makes the AI model deployment lifecycle different from a traditional software development lifecycle?
2. **CLO2**: When would you choose ONNX over Pickle? In what situations is Pickle sufficient?
3. **CLO3**: What is the difference between REST and gRPC for model serving? When would you choose each?
4. **CLO4**: If you needed to serve one million requests per day, what cloud deployment strategy would you choose?
5. **CLO5**: What is the difference between a Container and a Virtual Machine? Why are containers better suited for ML?
6. **CLO6**: If an A/B test shows the new model outperforms the old one by only 0.1%, would you promote it? What factors influence your decision?